# 13 RPPA Analysis and Multiomic Handoff

## Overview

This notebook performs strict 2-way RPPA analysis for downstream multiomic integration.
It ingests a prebuilt RPPA table from `RPPA/RPPA_data.csv`, enforces canonical NM0/OGM0 scope,
runs EDA/QC summaries, and exports NB14-ready handoff files.

## Analysis Scope
- Load preprocessed RPPA data from `RPPA/RPPA_data.csv`
- Enforce strict 2-way NM0/OGM0 analysis scope
- Generate summary heatmaps and delta views for inspection
- Export stable RPPA matrices and sample-mapping tables for Notebook 14

## Input Contract
- Required preprocessed RPPA table: `RPPA/RPPA_data.csv`
- Required metadata columns: `clean_id`, `group`, `batch`
- Optional metadata columns: `treatment`, `time`, `cage`
- Remaining numeric columns are treated as RPPA protein features

## Output Contract
- Plot outputs written to `Results/RPPA/` with notebook tag `nb13`
- Multiomic handoff tables written to `RPPA/for_multiomic/`
- Notebook 14 consumes the exported M0 Mock vs OG matrix plus sample-mapping metadata

## Handoff Files for Notebook 14
- `RPPA_nb14_ready_2way_m0_mockog.csv`
- `RPPA_nb14_sample_mapping_mockog.csv`

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid", context="talk")

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "Notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "Notebooks").resolve()
ANALYSIS_DIR = NOTEBOOK_DIR.parent.resolve()
OUT_DIR = ANALYSIS_DIR / "RPPA"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Central plot output contract for final notebook artifacts.
RESULTS_RPPA_DIR = ANALYSIS_DIR / "Results" / "RPPA"
RESULTS_RPPA_DIR.mkdir(parents=True, exist_ok=True)
NB_TAG = "13"

# Canonical 2-way groups for this notebook's strict analysis block.
GROUP_KEEP_2WAY = ["NM0", "OGM0"]
GROUP_ALIAS = {"OG0": "OGM0", "OG6": "OGM6"}


def save_svg(fig, stem):
    out_path = RESULTS_RPPA_DIR / f"{stem}_{NB_TAG}.svg"
    fig.savefig(out_path, format="svg", bbox_inches="tight")
    print(f"Saved: {out_path}")


print(f"Output folder: {OUT_DIR}")
print(f"Target groups (strict 2-way): {GROUP_KEEP_2WAY}")
print(f"Plot output folder: {RESULTS_RPPA_DIR}")

In [ ]:
META_EXPORT_COLS = [
    "clean_id",
    "group",
    "batch",
    "treatment",
    "time",
    "cage",
]

RPPA_DATA_PATH = ANALYSIS_DIR / "RPPA" / "RPPA_data.csv"


def load_rppa_data(path: Path):
    """Load canonical banked RPPA table used by Notebook 13 analysis."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required RPPA data file: {path}\n"
            "Create or copy RPPA_data.csv into RPPA/ before running Notebook 13."
        )

    df = pd.read_csv(path)

    # Backward compatibility with older banked files that still contained mouse_id.
    if "mouse_id" in df.columns:
        df = df.drop(columns=["mouse_id"])

    required = ["clean_id", "group", "batch"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"RPPA_data missing required columns: {missing}")

    for c in ["treatment", "time", "cage"]:
        if c not in df.columns:
            df[c] = "UNK"

    protein_cols = [c for c in df.columns if c not in META_EXPORT_COLS]
    if len(protein_cols) == 0:
        raise ValueError("RPPA_data has no protein feature columns after metadata columns are removed.")

    df[protein_cols] = df[protein_cols].apply(pd.to_numeric, errors="coerce")
    return df, protein_cols

In [ ]:
rppa_data_raw, protein_cols = load_rppa_data(RPPA_DATA_PATH)

# Normalize shorthand labels, then enforce strict 2-way filtering for this notebook.
rppa_data_raw["group"] = rppa_data_raw["group"].replace(GROUP_ALIAS)
rppa_data_2way = rppa_data_raw[rppa_data_raw["group"].isin(GROUP_KEEP_2WAY)].copy()

if rppa_data_2way.empty:
    raise ValueError(
        "No 2-way RPPA rows found after filtering for NM0 and OGM0. "
        "Check RPPA_data.csv group labels."
    )

present_groups = sorted(rppa_data_2way["group"].dropna().unique().tolist())
if any(g not in GROUP_KEEP_2WAY for g in present_groups):
    raise ValueError(f"Unexpected groups after 2-way filtering: {present_groups}")

# Build sample metadata indexed by clean_id for group filtering and export mapping.
sample_meta_bio = (
    rppa_data_2way[META_EXPORT_COLS]
    .drop_duplicates(subset=["clean_id"])
    .set_index("clean_id")
    .copy()
)

# Build protein x sample matrix expected by downstream notebook 13 cells.
expr_sample = rppa_data_2way.set_index("clean_id")[protein_cols].copy()
expr_sample = expr_sample.loc[sample_meta_bio.index].copy()
expr_bio = expr_sample.T.copy()
expr_bio.index.name = "Gene"

print(f"Loaded RPPA data file: {RPPA_DATA_PATH}")
print("Expression matrix (proteins x strict 2-way samples):", expr_bio.shape)
print("2-way group counts:")
print(sample_meta_bio["group"].value_counts())

## 2-Group Analysis (Mock vs OG, M0-only)

Reproduces the Notebook 14 RPPA EDA pipeline using only baseline (`M0`) samples for binary comparison.

**Group mapping used here (M0-only):**
- **Mock** = NM0
- **OG** = OGM0

**Notebook 13 EDA pipeline:**

| Step | Plot | Description |
|------|------|-------------|
| 1 | Histogram | Overall log2 intensity distribution across selected samples and proteins |
| 2 | Horizontal boxplot | Per-protein spread across selected samples |
| 3a | Z-heatmap | Protein z-scores; samples as rows, proteins as cols |
| 3b | PCA (2-group) | PC1/PC2 scatter, colored Mock/OG |
| 3c | Dendrogram | Ward-linkage hierarchical clustering of samples |
| 4 | Sample-corr heatmap | Pearson correlation between all sample pairs |
| 5 | Differential: t-test | Per-protein two-sided t-test (OG vs Mock), BH-FDR correction |
| 6a | Volcano plot | log2FC vs -log10(p); red = FDR < 0.05 |
| 6b | Sig-protein heatmap | Z-scored heatmap of proteins with FDR < 0.05 and |log2FC| > 0.5 |
| 7 | Strip+boxplot grid | Per-significant-protein strip+box for Mock vs OG comparison |

The cells below implement steps 1-7 using M0 samples.

In [ ]:
from scipy.cluster.hierarchy import linkage as _hier_linkage, dendrogram as _hier_dendrogram
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.decomposition import PCA as _PCA

# Build sample-level log2 matrix from strict 2-way biological expression matrix.
expr_log2 = np.log2(expr_sample + 1)


def _protein_label(idx):
    if isinstance(idx, tuple):
        if len(idx) >= 4:
            return str(idx[3])
        return str(idx[-1])
    return str(idx)


# ---- Build named (string column) expression matrix ----
_pl = [_protein_label(i) for i in expr_log2.columns]
expr_named_2grp = expr_log2.copy()
expr_named_2grp.columns = _pl
# Collapse any duplicate gene symbols by mean (matches NB14 preprocessing)
expr_named_2grp = expr_named_2grp.T.groupby(level=0).mean().T

# ---- 2-group assignment: Mock = NM0, OG = OGM0 ----
group2_vec = sample_meta_bio.loc[expr_named_2grp.index, "group"].map({"NM0": "Mock", "OGM0": "OG"})
if group2_vec.isna().any():
    bad = group2_vec[group2_vec.isna()].index.tolist()
    raise ValueError(f"Unexpected sample labels in 2-group block: {bad[:5]}")
_grp2_colors = {"Mock": "skyblue", "OG": "salmon"}

print("2-group block uses strict NM0/OGM0 samples:")
print(group2_vec.value_counts())

### 2-Group Plot 1: Distribution Histogram (M0-only)

In [ ]:
# ---------- 1. Distribution histogram ----------
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(expr_named_2grp.values.flatten(), bins=50, kde=True, ax=ax, color="steelblue", alpha=0.7)
ax.set_xlabel("log2(RPPA intensity + 1)")
ax.set_ylabel("Frequency")
ax.set_title("Overall RPPA distribution (M0-only samples)")
plt.tight_layout()
save_svg(fig, "rppa_2group_m0_distribution_hist")
plt.show()

### 2-Group Plot 2: Per-Protein Spread Boxplot (M0-only)

In [ ]:
# ---------- 2. Per-protein spread boxplot ----------
n_prot = expr_named_2grp.shape[1]
fig, ax = plt.subplots(figsize=(10, max(8, 0.28 * n_prot)))
sns.boxplot(data=expr_named_2grp, orient="h", palette="Set3", ax=ax)
ax.set_xlabel("log2(RPPA intensity + 1)")
ax.set_ylabel("Protein")
ax.set_title("Protein expression spread across M0-only samples")
plt.tight_layout()
save_svg(fig, "rppa_2group_m0_protein_spread_boxplot")
plt.show()

### 2-Group Plot 3: Z-Heatmap, PCA, and Dendrogram (M0-only)

In [ ]:
# ---------- 3-panel: z-heatmap + PCA + dendrogram ----------
_df_z = (expr_named_2grp - expr_named_2grp.mean()) / expr_named_2grp.std()
_df_z_disp = _df_z.copy()
_df_z_disp.index = (group2_vec.values + " (" + expr_named_2grp.index.astype(str) + ")")

_X_sc = _SS().fit_transform(expr_named_2grp.fillna(0))
_pca = _PCA(n_components=2)
_pcs = _pca.fit_transform(_X_sc)
df_pca_2grp = pd.DataFrame(_pcs, columns=["PC1", "PC2"], index=expr_named_2grp.index)
df_pca_2grp["Group"] = group2_vec.values

_Z_link = _hier_linkage(expr_named_2grp.fillna(0), method="ward")
_dend_labels = (group2_vec.values + " (" + expr_named_2grp.index.astype(str) + ")")

fig, axes = plt.subplots(1, 3, figsize=(22, 8))

sns.heatmap(
    _df_z_disp, cmap="vlag", linewidths=0.0,
    cbar_kws={"label": "z-score"}, ax=axes[0],
    yticklabels=True, xticklabels=False,
)
axes[0].set_title("Z-heatmap (M0-only samples x proteins)")
axes[0].set_xlabel("Proteins")
axes[0].set_ylabel("Samples")
axes[0].text(-0.2, 1.05, "a.", transform=axes[0].transAxes, fontsize=16, fontweight="bold", va="top")

sns.scatterplot(
    data=df_pca_2grp, x="PC1", y="PC2", hue="Group",
    palette=_grp2_colors, s=120, ax=axes[1],
)
for sid, row in df_pca_2grp.iterrows():
    axes[1].text(row["PC1"] + 0.05, row["PC2"] + 0.05, str(sid), fontsize=8)
axes[1].set_title("PCA of RPPA samples (2-group, M0-only)")
axes[1].set_xlabel(f"PC1 ({_pca.explained_variance_ratio_[0]*100:.1f}% var)")
axes[1].set_ylabel(f"PC2 ({_pca.explained_variance_ratio_[1]*100:.1f}% var)")
axes[1].text(-0.2, 1.05, "b.", transform=axes[1].transAxes, fontsize=16, fontweight="bold", va="top")

_hier_dendrogram(_Z_link, labels=_dend_labels, ax=axes[2])
axes[2].set_title("Hierarchical clustering (Ward, M0-only)")
axes[2].set_ylabel("Distance")
axes[2].text(-0.2, 1.05, "c.", transform=axes[2].transAxes, fontsize=16, fontweight="bold", va="top")

plt.tight_layout()
save_svg(fig, "rppa_2group_m0_heatmap_pca_dendrogram")
plt.show()

### 2-Group Plot 4: Sample-to-Sample Pearson Correlation (M0-only)

In [ ]:
# ---------- 4. Sample-to-sample Pearson correlation ----------
_sample_corr = expr_named_2grp.T.corr()
_tick_labels = (group2_vec.values + "\n" + expr_named_2grp.index.astype(str))
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    _sample_corr, cmap="coolwarm", vmin=0.8, vmax=1.0,
    xticklabels=_tick_labels, yticklabels=_tick_labels, ax=ax,
)
ax.set_title("Sample-to-sample Pearson correlation (RPPA, M0-only)")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout()
save_svg(fig, "rppa_2group_m0_sample_correlation")
plt.show()

In [ ]:

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# ---- Low-variance filter (mirrors NB14 threshold) ----
variance_2grp = expr_named_2grp.var()
df_filtered_2grp = expr_named_2grp.loc[:, variance_2grp > 0.01]

mock_idx = group2_vec[group2_vec == "Mock"].index
og_idx = group2_vec[group2_vec == "OG"].index
mock_df_2grp = df_filtered_2grp.loc[mock_idx]
og_df_2grp = df_filtered_2grp.loc[og_idx]

# ---- t-test per protein ----
results_2grp = []
for protein in df_filtered_2grp.columns:
    m_vals = mock_df_2grp[protein].dropna()
    o_vals = og_df_2grp[protein].dropna()
    if len(m_vals) < 2 or len(o_vals) < 2:
        results_2grp.append({"Protein": protein, "log2FC": np.nan, "pvalue": np.nan})
        continue
    stat, pval = ttest_ind(o_vals, m_vals)
    log2fc = o_vals.mean() - m_vals.mean()
    results_2grp.append({"Protein": protein, "log2FC": log2fc, "pvalue": pval})

results_df_2grp = pd.DataFrame(results_2grp)
results_df_2grp["pvalue"] = results_df_2grp["pvalue"].fillna(1.0).replace(0, 1e-300)
valid_mask = results_df_2grp["pvalue"].notna() & results_df_2grp["log2FC"].notna()
results_df_2grp = results_df_2grp[valid_mask].copy()
results_df_2grp["FDR"] = multipletests(results_df_2grp["pvalue"], method="fdr_bh")[1]

sig_mask_2grp = (results_df_2grp["FDR"] < 0.05) & (results_df_2grp["log2FC"].abs() > 0.5)
sig_proteins_2grp = results_df_2grp.loc[sig_mask_2grp, "Protein"].tolist()
print(f"Significant proteins (FDR < 0.05, |log2FC| > 0.5): {len(sig_proteins_2grp)}")
display(results_df_2grp.sort_values("FDR").head(20))

# ---- Figure 1: Volcano + sig-protein heatmap ----
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax_v = axes[0]
is_sig_2grp = results_df_2grp["FDR"] < 0.05
sns.scatterplot(
    data=results_df_2grp,
    x="log2FC",
    y=-np.log10(results_df_2grp["pvalue"]),
    hue=is_sig_2grp,
    palette={True: "red", False: "gray"},
    alpha=0.75,
    ax=ax_v,
)
ax_v.axhline(-np.log10(0.05), linestyle="--", color="blue", alpha=0.5)
ax_v.axvline(0, linestyle="--", color="black", alpha=0.4)
ax_v.set_xlabel("log2FC (OG - Mock)")
ax_v.set_ylabel("-log10(p-value)")
ax_v.set_title("Volcano: OG vs Mock")
ax_v.text(-0.1, 1.05, "a.", transform=ax_v.transAxes, fontsize=16, fontweight="bold", va="top")

if len(sig_proteins_2grp) > 0:
    df_sig_2grp = df_filtered_2grp[sig_proteins_2grp].copy()
    df_sig_z = (df_sig_2grp - df_sig_2grp.mean()) / df_sig_2grp.std()
    sns.heatmap(
        df_sig_z, cmap="vlag", linewidths=0.5,
        cbar_kws={"label": "z-score"}, ax=axes[1],
    )
    axes[1].set_title("Significant proteins (FDR < 0.05, |log2FC| > 0.5)")
    axes[1].set_xlabel("Proteins")
    axes[1].set_ylabel("Samples")
    # Make sample names readable by drawing y tick labels horizontally.
    axes[1].tick_params(axis="y", rotation=0, labelsize=10)
    axes[1].text(-0.05, 1.05, "b.", transform=axes[1].transAxes, fontsize=16, fontweight="bold", va="top")
else:
    axes[1].text(0.5, 0.5, "No significant proteins", ha="center", va="center", fontsize=14)
    axes[1].set_axis_off()

plt.tight_layout()
save_svg(fig, "rppa_2group_m0_volcano_sig_heatmap")
plt.show()


In [ ]:

# ---- Per-significant-protein strip+boxplot grid (replicates NB14 Figure 2) ----
if len(sig_proteins_2grp) > 0:
    n_cols = 4
    n_rows = int(np.ceil(len(sig_proteins_2grp) / n_cols))
    fig, axes_bp = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
    axes_bp = np.array(axes_bp).flatten()

    for i, protein in enumerate(sig_proteins_2grp):
        ax = axes_bp[i]
        plot_df = df_filtered_2grp[[protein]].copy()
        plot_df["Group"] = group2_vec.values
        sns.stripplot(
            x="Group", y=protein, data=plot_df, jitter=True,
            ax=ax, palette=_grp2_colors, hue="Group", dodge=False, size=5,
        )
        sns.boxplot(
            x="Group", y=protein, data=plot_df, showcaps=False,
            boxprops={"facecolor": "None"}, ax=ax,
        )
        ax.set_title(protein)
        lg = ax.get_legend()
        if lg is not None:
            lg.remove()

    for j in range(len(sig_proteins_2grp), len(axes_bp)):
        fig.delaxes(axes_bp[j])

    plt.suptitle("Significant proteins: OG vs Mock (strip+box)", y=1.01)
    plt.tight_layout()
    save_svg(fig, "rppa_2group_m0_significant_protein_grid")
    plt.show()
else:
    print("No significant proteins to plot.")


In [ ]:
EXPORT_DIR = OUT_DIR / "for_multiomic"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Build Notebook-14-compatible RPPA exports from strict 2-way sample-level matrix.
# Contract for NB14: columns must be sample IDs with Mock/OG prefixes.

sample_export_meta = sample_meta_bio.copy()
sample_export_meta["treatment_2lvl"] = sample_export_meta["group"].map({"NM0": "Mock", "OGM0": "OG"})
sample_export_meta["time_2lvl"] = "M0"

# Stable ordering keeps exports deterministic across runs.
sample_export_meta = sample_export_meta.sort_values(
    ["treatment_2lvl", "batch", "clean_id"]
).copy()

# Pull gene symbols from RPPA index and collapse duplicate genes by mean.
if isinstance(expr_bio.index, pd.MultiIndex) and "Gene_ID" in expr_bio.index.names:
    gene_labels = expr_bio.index.get_level_values("Gene_ID").astype(str)
else:
    gene_labels = pd.Index(expr_bio.index.astype(str), name="Gene")


def sort_mock_og_ids(sample_ids):
    """Sort IDs as Mock-1..N then OG-1..N by numeric suffix."""
    _ord = {"Mock": 0, "OG": 1}

    def _key(x):
        s = str(x)
        m = re.match(r"^(Mock|OG)-(\d+)$", s)
        if not m:
            return (99, 10**9, s)
        return (_ord[m.group(1)], int(m.group(2)), s)

    return sorted(sample_ids, key=_key)


def build_export(ids, rank_col_name, id_col_name):
    """Build gene x sample export with Mock/OG-style sample IDs."""
    sub_meta = sample_export_meta.loc[ids].copy()
    sub_meta[rank_col_name] = sub_meta.groupby("treatment_2lvl").cumcount() + 1
    sub_meta[id_col_name] = sub_meta["treatment_2lvl"] + "-" + sub_meta[rank_col_name].astype(str)

    x = expr_bio.loc[:, ids].copy()
    x.columns = sub_meta[id_col_name].values
    x.index.name = None
    x["Gene"] = gene_labels.values

    out = x.groupby("Gene", as_index=False).mean(numeric_only=True)
    cols = [c for c in out.columns if c != "Gene"]
    cols = sort_mock_og_ids(cols)
    out = out[["Gene"] + cols]
    return out, sub_meta


ids_2way = sample_export_meta.index.tolist()
rppa_nb14_m0, meta_m0 = build_export(
    ids_2way,
    rank_col_name="sample_rank_treatment_m0",
    id_col_name="sample_id_nb14_m0",
)


def preview_nb14_contract(df_export, name):
    df_rppa = df_export.set_index("Gene").T
    df_rppa.index = df_rppa.index.str.replace("_", "-", regex=False)
    df_rppa_numeric = df_rppa.apply(pd.to_numeric, errors="coerce")
    df_log = np.log2(df_rppa_numeric + 1)

    print(f"[{name}] raw export shape (genes x samples): {df_export.shape}")
    print(f"[{name}] notebook14 df_log shape (samples x genes): {df_log.shape}")
    print(f"[{name}] sample IDs preview: {df_log.index.tolist()[:6]}")
    print(f"[{name}] sample counts:")
    print(pd.Series(["Mock" if s.startswith("Mock-") else "OG" for s in df_log.index]).value_counts().to_string())
    print()


preview_nb14_contract(rppa_nb14_m0, "M0_2WAY_MOCK_OG")

# Mapping table for traceability from clean IDs to exported IDs.
export_meta = sample_export_meta.reset_index().rename(columns={"index": "clean_id"})
export_meta = export_meta.merge(
    meta_m0[["sample_id_nb14_m0"]],
    left_on="clean_id",
    right_index=True,
    how="left",
)

export_meta = export_meta[[
    "clean_id",
    "group",
    "treatment_2lvl",
    "time_2lvl",
    "batch",
    "sample_id_nb14_m0",
]].copy()

print("Export mapping preview:")
display(export_meta.head(12))

WRITE_MULTIOMIC_EXPORTS = True
if WRITE_MULTIOMIC_EXPORTS:
    rppa_nb14_m0.to_csv(EXPORT_DIR / "RPPA_nb14_ready_2way_m0_mockog.csv", index=False)
    export_meta.to_csv(EXPORT_DIR / "RPPA_nb14_sample_mapping_mockog.csv", index=False)
    print(f"Wrote exports to: {EXPORT_DIR}")
else:
    print("WRITE_MULTIOMIC_EXPORTS=False; no files written yet.")
    print(f"Planned export directory: {EXPORT_DIR}")

## Multiomic Export Prep (Notebook 14-Compatible)

This section keeps RPPA at sample level for EDA, then builds strict 2-way exports that can be loaded directly by Notebook 14.

Exports prepared:
- `M0-only` candidate: NM0 mapped to `Mock-*`, OGM0 mapped to `OG-*`
- sample metadata mapping table for traceability (`clean_id` -> exported sample id)

Notes:
- RPPA values stay in normalized intensity space (Notebook 14 applies `log2(x+1)` internally).
- Gene symbols are taken from `Gene` and duplicate genes are collapsed by mean to match Notebook 14 feature expectations.

## Next Step (Optional)

After validating this final notebook, you can archive or delete older RPPA notebooks:
- `RPPA.ipynb`
- `RPPA_raw_qi_pipeline.ipynb`
- `RPPA_raw_qi_pipeline_2.ipynb`